In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# Find exact path
import os
for root, dirs, files in os.walk('/kaggle/input/'):
    print(f"Directory: {root}")
    print(f"  Subdirs: {dirs[:5]}")
    print(f"  Files (first 3): {files[:3]}")
    print(f"  Total files here: {len(files)}")
    print()
    if root.count('/') > 5:  # Don't go too deep
        break

In [ ]:
# verify
import glob
psv_path = '/kaggle/input/datasets/salikhussaini49/prediction-of-sepsis/training_setA/training'
files = glob.glob(f'{psv_path}/*.psv')
print(f"PSV files found: {len(files):,}")
print(f"First 3 files: {files[:3]}")

In [ ]:
# Install libraries
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             roc_curve, precision_recall_curve,
                             classification_report, brier_score_loss)
from sklearn.calibration import calibration_curve

import xgboost as xgb
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

!pip install captum -q

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"GPU available: {torch.cuda.is_available()}")

os.makedirs('images', exist_ok=True)

In [ ]:
#Preprocessing cell
# Load data
def load_patient_files(data_dir, limit=None):
    files = sorted(glob.glob(os.path.join(data_dir, '*.psv')))
    if limit:
        files = files[:limit]
    dfs = []
    for f in tqdm(files, desc='Loading patients'):
        d = pd.read_csv(f, sep='|')
        d['patient_id'] = os.path.basename(f).replace('.psv', '')
        dfs.append(d)
    return pd.concat(dfs, ignore_index=True)

DATA_PATH = '/kaggle/input/datasets/salikhussaini49/prediction-of-sepsis/training_setA/training'
df = load_patient_files(DATA_PATH)
print(f"\nLoaded {df['patient_id'].nunique():,} patients, {len(df):,} records")

# Smart feature selection based on patient-level availability
feature_cols_all = [c for c in df.columns
                    if c not in ['patient_id', 'SepsisLabel', 'ICULOS']]
patient_availability = df.groupby('patient_id')[feature_cols_all].apply(
    lambda x: x.notna().any()).mean().sort_values(ascending=False)

AVAILABILITY_THRESHOLD = 0.5
features_to_keep = patient_availability[
    patient_availability >= AVAILABILITY_THRESHOLD].index.tolist()

# Preprocessing with imputation
final_features = features_to_keep
if 'ICULOS' not in final_features:
    final_features = final_features + ['ICULOS']

df_filtered = df[['patient_id', 'SepsisLabel'] + final_features].copy()
df_filtered = df_filtered.sort_values(['patient_id', 'ICULOS'])
df_filtered[final_features] = df_filtered.groupby('patient_id')[final_features].ffill()
df_filtered[final_features] = df_filtered.groupby('patient_id')[final_features].bfill()
for col in final_features:
    df_filtered[col] = df_filtered[col].fillna(df_filtered[col].median())

print(f"Features: {len(final_features)}, Missing after imputation: {df_filtered[final_features].isnull().sum().sum()}")

# Time window creation
WINDOW_SIZE = 12
PREDICTION_HORIZON = 6

def create_sequences(df, window_size, horizon, features):
    sequences, labels, patient_ids, timestamps = [], [], [], []
    grouped = df.groupby('patient_id')
    for patient_id, p_data in tqdm(grouped, desc='Building sequences'):
        p_data = p_data.sort_values('ICULOS').reset_index(drop=True)
        if len(p_data) < window_size + horizon:
            continue
        feat_array = p_data[features].values
        labels_array = p_data['SepsisLabel'].values
        for t in range(window_size, len(p_data) - horizon):
            sequences.append(feat_array[t - window_size:t])
            labels.append(int(labels_array[t:t + horizon].max()))
            patient_ids.append(patient_id)
            timestamps.append(t)
    return (np.array(sequences, dtype=np.float32),
            np.array(labels, dtype=np.float32),
            np.array(patient_ids), np.array(timestamps))

X, y, pids, times = create_sequences(df_filtered, WINDOW_SIZE, PREDICTION_HORIZON, final_features)

# Patient-level split
unique_patients = np.unique(pids)
train_patients, test_patients = train_test_split(unique_patients, test_size=0.2, random_state=42)
train_mask = np.isin(pids, train_patients)
test_mask = np.isin(pids, test_patients)

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]
pids_test = pids[test_mask]
times_test = times[test_mask]

n_train, n_steps, n_features = X_train.shape
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.reshape(-1, n_features)).reshape(n_train, n_steps, n_features)
X_test_scaled = scaler.transform(X_test.reshape(-1, n_features)).reshape(-1, n_steps, n_features)

X_train_flat = X_train_scaled.reshape(len(X_train_scaled), -1)
X_test_flat = X_test_scaled.reshape(len(X_test_scaled), -1)

print(f"\n✅ All preprocessing complete")
print(f"Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}")
print(f"Train positive rate: {y_train.mean():.2%}, Test positive rate: {y_test.mean():.2%}")

In [ ]:
# EDA 1 — Sepsis prevalence and stay distribution
sepsis_patients = df.groupby('patient_id')['SepsisLabel'].max()
stay_lengths = df.groupby('patient_id')['ICULOS'].max()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sepsis_counts = sepsis_patients.value_counts()
axes[0].bar(['No Sepsis', 'Sepsis'], sepsis_counts.values,
            color=['steelblue', 'coral'])
axes[0].set_title('Sepsis Prevalence in Dataset')
axes[0].set_ylabel('Number of Patients')
for i, v in enumerate(sepsis_counts.values):
    axes[0].text(i, v + len(sepsis_patients) * 0.01,
                 f'{v:,}\n({v/len(sepsis_patients):.1%})',
                 ha='center', fontweight='bold')

axes[1].hist(stay_lengths.values, bins=50, color='steelblue', edgecolor='white')
axes[1].set_title('ICU Stay Length Distribution')
axes[1].set_xlabel('Hours in ICU')
axes[1].set_ylabel('Number of Patients')
axes[1].axvline(stay_lengths.median(), color='red', linestyle='--',
                label=f'Median: {stay_lengths.median():.0f}h')
axes[1].legend()

plt.tight_layout()
plt.savefig('images/sepsis_prevalence.png', dpi=150, bbox_inches='tight')
plt.show()

# EDA 2 — Missing data analysis
missing_rates = df[feature_cols_all].isnull().mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(16, 8))
colors = ['coral' if r > 0.6 else 'orange' if r > 0.3 else 'steelblue'
          for r in missing_rates.values]
ax.bar(range(len(missing_rates)), missing_rates.values, color=colors)
ax.set_xticks(range(len(missing_rates)))
ax.set_xticklabels(missing_rates.index, rotation=45, ha='right')
ax.set_ylabel('Missing Rate')
ax.set_title('Missing Data Rate by Clinical Feature')
ax.axhline(y=0.6, color='red', linestyle='--', alpha=0.5, label='60% threshold')
ax.axhline(y=0.3, color='orange', linestyle='--', alpha=0.5, label='30% threshold')
ax.legend()
plt.tight_layout()
plt.savefig('images/missing_data_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# EDA 3 — Vital signs trajectory
vitals = ['HR', 'O2Sat', 'Temp', 'SBP', 'Resp', 'MAP']
vital_names = ['Heart Rate (bpm)', 'O2 Saturation (%)', 'Temperature (°C)',
               'Systolic BP (mmHg)', 'Respiratory Rate', 'Mean Arterial Pressure']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, vital, name in zip(axes.flatten(), vitals, vital_names):
    for label, color, ls in [(0, 'steelblue', '-'), (1, 'coral', '--')]:
        subset = df[df['SepsisLabel'] == label]
        mean_vals = subset.groupby('ICULOS')[vital].mean()
        std_vals = subset.groupby('ICULOS')[vital].std()
        hours = mean_vals.index[:72]
        mean = mean_vals.values[:72]
        std = std_vals.values[:72]
        label_str = 'Sepsis' if label == 1 else 'No Sepsis'
        ax.plot(hours, mean, color=color, linestyle=ls, label=label_str, linewidth=2)
        ax.fill_between(hours, mean - std, mean + std, alpha=0.15, color=color)
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Hours in ICU')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Vital Signs Trajectories: Sepsis vs Non-Sepsis Patients',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('images/vital_signs_over_time.png', dpi=150, bbox_inches='tight')
plt.show()

# Also need feature_cols_all for the missing rates plot to work
print("\n✅ All EDA plots saved to /kaggle/working/images/")

In [ ]:
# Baselines (Logistic Regression + XGBoost)

# Logistic Regression baseline
print("Training Logistic Regression...")
lr_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr_model.fit(X_train_flat, y_train)
lr_prob = lr_model.predict_proba(X_test_flat)[:, 1]
lr_auc = roc_auc_score(y_test, lr_prob)
lr_ap = average_precision_score(y_test, lr_prob)
print(f"Logistic Regression — AUC: {lr_auc:.4f}, AP: {lr_ap:.4f}")

# XGBoost baseline
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print("\nTraining XGBoost...")
xgb_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    scale_pos_weight=scale_pos_weight, random_state=42,
    eval_metric='logloss', tree_method='hist', device='cuda')
xgb_model.fit(X_train_flat, y_train)
xgb_prob = xgb_model.predict_proba(X_test_flat)[:, 1]
xgb_auc = roc_auc_score(y_test, xgb_prob)
xgb_ap = average_precision_score(y_test, xgb_prob)
print(f"XGBoost — AUC: {xgb_auc:.4f}, AP: {xgb_ap:.4f}")

In [ ]:
# The Attention LSTM 
# Define the Attention LSTM
class AttentionLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.3):
        super(AttentionLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )

        self.attention = nn.Sequential(
            nn.Linear(hidden_size * 2, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, x, return_attention=False):
        lstm_out, _ = self.lstm(x)
        attn_scores = self.attention(lstm_out).squeeze(-1)
        attn_weights = F.softmax(attn_scores, dim=1)
        context = torch.bmm(attn_weights.unsqueeze(1), lstm_out).squeeze(1)
        logits = self.classifier(context).squeeze(-1)
        if return_attention:
            return logits, attn_weights
        return logits

# Dataset class
class SepsisDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = SepsisDataset(X_train_scaled, y_train)
test_ds = SepsisDataset(X_test_scaled, y_test)

# Weighted sampling for class imbalance
class_counts = np.array([(y_train == 0).sum(), (y_train == 1).sum()])
weights = 1.0 / class_counts
sample_weights = weights[y_train.astype(int)]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=256, sampler=sampler)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

# Initialise model
model = AttentionLSTM(
    input_size=len(final_features),
    hidden_size=128,
    num_layers=2,
    dropout=0.3
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")

# Training setup
pos_weight = torch.tensor([(y_train == 0).sum() / (y_train == 1).sum()]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

def train_epoch(model, loader):
    model.train()
    total_loss = 0
    preds, labels = [], []
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        preds.extend(torch.sigmoid(logits).detach().cpu().numpy())
        labels.extend(y_batch.cpu().numpy())
    return total_loss / len(loader), roc_auc_score(labels, preds)

def evaluate(model, loader):
    model.eval()
    total_loss = 0
    preds, labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            total_loss += loss.item()
            preds.extend(torch.sigmoid(logits).cpu().numpy())
            labels.extend(y_batch.cpu().numpy())
    return total_loss / len(loader), roc_auc_score(labels, preds), np.array(preds), np.array(labels)

# Training loop
NUM_EPOCHS = 25
best_val_auc = 0
train_losses, val_losses = [], []
train_aucs, val_aucs = [], []

print("\nTraining Attention LSTM...")
for epoch in range(NUM_EPOCHS):
    train_loss, train_auc = train_epoch(model, train_loader)
    val_loss, val_auc, _, _ = evaluate(model, test_loader)
    scheduler.step(val_loss)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_aucs.append(train_auc)
    val_aucs.append(val_auc)

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(model.state_dict(), 'best_model.pt')

    print(f"Epoch {epoch+1:2d}/{NUM_EPOCHS} | Train AUC: {train_auc:.4f} | Val AUC: {val_auc:.4f}")

print(f"\n✅ Best Validation AUC: {best_val_auc:.4f}")

In [ ]:
# Reset the model
model = AttentionLSTM(
    input_size=len(final_features),
    hidden_size=128,
    num_layers=2,
    dropout=0.5  # Increased dropout
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")

# Use regular shuffled loader (no weighted sampling)
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

# Class imbalance handled through loss weighting only
pos_weight = torch.tensor([(y_train == 0).sum() / (y_train == 1).sum()]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Lower learning rate and stronger weight decay for regularisation
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

# Early stopping
NUM_EPOCHS = 20
best_val_auc = 0
patience_counter = 0
EARLY_STOP_PATIENCE = 4

train_losses, val_losses = [], []
train_aucs, val_aucs = [], []

print("\nTraining Attention LSTM (with better regularisation)...")
for epoch in range(NUM_EPOCHS):
    train_loss, train_auc = train_epoch(model, train_loader)
    val_loss, val_auc, _, _ = evaluate(model, test_loader)
    scheduler.step(val_loss)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_aucs.append(train_auc)
    val_aucs.append(val_auc)

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(model.state_dict(), 'best_model.pt')
        patience_counter = 0
        marker = " ⭐ (best)"
    else:
        patience_counter += 1
        marker = ""

    print(f"Epoch {epoch+1:2d}/{NUM_EPOCHS} | Train AUC: {train_auc:.4f} | Val AUC: {val_auc:.4f}{marker}")

    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f"\n⏹️ Early stopping — no improvement for {EARLY_STOP_PATIENCE} epochs")
        break

print(f"\n✅ Best Validation AUC: {best_val_auc:.4f}")

In [ ]:
# Load best model and generate final predictions
model.load_state_dict(torch.load('best_model.pt'))
_, lstm_auc, lstm_prob, _ = evaluate(model, test_loader)
lstm_ap = average_precision_score(y_test, lstm_prob)

print("="*60)
print("FINAL MODEL COMPARISON")
print("="*60)
print(f"{'Model':<25} {'AUC-ROC':>10} {'AP':>10}")
print("-"*60)
print(f"{'Logistic Regression':<25} {lr_auc:>10.4f} {lr_ap:>10.4f}")
print(f"{'XGBoost':<25} {xgb_auc:>10.4f} {xgb_ap:>10.4f}")
print(f"{'Attention LSTM':<25} {lstm_auc:>10.4f} {lstm_ap:>10.4f}")
print("="*60)

# Training curves visualisation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_losses, label='Train Loss', color='steelblue')
axes[0].plot(val_losses, label='Validation Loss', color='coral')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(train_aucs, label='Train AUC', color='steelblue')
axes[1].plot(val_aucs, label='Validation AUC', color='coral')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('AUC-ROC')
axes[1].set_title('Training & Validation AUC')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('images/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# ROC and PR curves comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for name, prob, auc, color in [
    ('Logistic Regression', lr_prob, lr_auc, 'steelblue'),
    ('XGBoost', xgb_prob, xgb_auc, 'orange'),
    ('Attention LSTM', lstm_prob, lstm_auc, 'coral')]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    axes[0].plot(fpr, tpr, color=color, linewidth=2,
                 label=f'{name} (AUC = {auc:.3f})')

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random baseline')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve Comparison')
axe

In [ ]:
# ROC and PR curves comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for name, prob, auc, color in [
    ('Logistic Regression', lr_prob, lr_auc, 'steelblue'),
    ('XGBoost', xgb_prob, xgb_auc, 'orange'),
    ('Attention LSTM', lstm_prob, lstm_auc, 'coral')]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    axes[0].plot(fpr, tpr, color=color, linewidth=2,
                 label=f'{name} (AUC = {auc:.3f})')

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random baseline')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve Comparison')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

for name, prob, ap, color in [
    ('Logistic Regression', lr_prob, lr_ap, 'steelblue'),
    ('XGBoost', xgb_prob, xgb_ap, 'orange'),
    ('Attention LSTM', lstm_prob, lstm_ap, 'coral')]:
    precision, recall, _ = precision_recall_curve(y_test, prob)
    axes[1].plot(recall, precision, color=color, linewidth=2,
                 label=f'{name} (AP = {ap:.3f})')

axes[1].axhline(y=y_test.mean(), color='k', linestyle='--', alpha=0.5,
                label=f'Random ({y_test.mean():.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve Comparison')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('images/model_comparison_roc.png', dpi=150, bbox_inches='tight')
plt.show()

# Calibration plot
fig, ax = plt.subplots(figsize=(8, 8))

for name, prob, color in [
    ('Logistic Regression', lr_prob, 'steelblue'),
    ('XGBoost', xgb_prob, 'orange'),
    ('Attention LSTM', lstm_prob, 'coral')]:
    fraction_pos, mean_pred = calibration_curve(y_test, prob, n_bins=10)
    brier = brier_score_loss(y_test, prob)
    ax.plot(mean_pred, fraction_pos, marker='o', color=color,
            linewidth=2, label=f'{name} (Brier: {brier:.4f})')

ax.plot([0, 1], [0, 1], 'k--', label='Perfectly calibrated')
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.set_title('Model Calibration — Reliability Diagram')
ax.legend()
ax.grid(True, alpha=0.3)
plt.savefig('images/calibration_plot.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Plots complete")

In [ ]:
# Attention visualisations
# Get attention weights function
def get_attention_weights(model, X_batch):
    model.eval()
    with torch.no_grad():
        X_tensor = torch.FloatTensor(X_batch).to(device)
        logits, attn_weights = model(X_tensor, return_attention=True)
        probs = torch.sigmoid(logits).cpu().numpy()
        return probs, attn_weights.cpu().numpy()

# Find high-confidence true positive and true negative cases
y_pred = (lstm_prob >= 0.5).astype(int)
tp_indices = np.where((y_pred == 1) & (y_test == 1))[0]
tn_indices = np.where((y_pred == 0) & (y_test == 0))[0]

print(f"True positives available: {len(tp_indices)}")
print(f"True negatives available: {len(tn_indices)}")

# Pick highest-confidence examples
if len(tp_indices) > 0:
    tp_confs = lstm_prob[tp_indices]
    sample_tp_idx = tp_indices[np.argmax(tp_confs)]
else:
    # Fall back to highest-probability sepsis case
    sample_tp_idx = np.argmax(lstm_prob * y_test)

sample_tn_idx = tn_indices[np.argmin(lstm_prob[tn_indices])]

# Attention heatmap plotting function
def plot_attention_heatmap(sequence, attn_weights, feature_names, title, save_path):
    fig, axes = plt.subplots(2, 1, figsize=(16, 12),
                              gridspec_kw={'height_ratios': [1, 4]})

    axes[0].bar(range(len(attn_weights)), attn_weights,
                color='coral', alpha=0.8)
    axes[0].set_title(f'Attention Weights Over Time — {title}',
                      fontweight='bold', fontsize=13)
    axes[0].set_ylabel('Attention Weight')
    axes[0].set_xticks(range(len(attn_weights)))
    axes[0].set_xticklabels([f't-{WINDOW_SIZE-i}' for i in range(WINDOW_SIZE)])

    sns.heatmap(sequence.T, ax=axes[1], cmap='RdBu_r', center=0,
                yticklabels=feature_names,
                cbar_kws={'label': 'Normalised Value'},
                xticklabels=[f't-{WINDOW_SIZE-i}' for i in range(WINDOW_SIZE)])
    axes[1].set_title('Feature Values Over the 12-Hour Window')
    axes[1].set_xlabel('Time Step')

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

# Sepsis case
sample_X = X_test_scaled[sample_tp_idx:sample_tp_idx+1]
prob, attn = get_attention_weights(model, sample_X)
print(f"\nSepsis case — Predicted probability: {prob[0]:.3f}")
plot_attention_heatmap(
    sample_X[0], attn[0], final_features,
    f"Sepsis Patient (Prediction: {prob[0]:.2f})",
    'images/attention_heatmap_sepsis.png')

# Non-sepsis case
sample_X = X_test_scaled[sample_tn_idx:sample_tn_idx+1]
prob, attn = get_attention_weights(model, sample_X)
print(f"Non-sepsis case — Predicted probability: {prob[0]:.3f}")
plot_attention_heatmap(
    sample_X[0], attn[0], final_features,
    f"Non-Sepsis Patient (Prediction: {prob[0]:.2f})",
    'images/attention_heatmap_no_sepsis.png')

print("\n✅ Attention heatmaps saved")

In [ ]:
from captum.attr import IntegratedGradients

# Wrap model for Captum — set to train mode for gradient computation
class WrappedModel(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        return torch.sigmoid(self.model(x))

wrapped = WrappedModel(model).to(device)
wrapped.train()  # ← Required for cuDNN LSTM gradient computation

ig = IntegratedGradients(wrapped)

# Compute attributions across sepsis cases
n_samples = min(100, len(tp_indices))
sample_indices = np.random.choice(tp_indices, n_samples, replace=False)

sample_X = torch.FloatTensor(X_test_scaled[sample_indices]).to(device)
sample_X.requires_grad = True
baseline = torch.zeros_like(sample_X).to(device)

print("Computing feature attributions with Integrated Gradients...")
attributions = ig.attribute(sample_X, baseline, n_steps=50)
attributions = attributions.detach().cpu().numpy()

# Aggregate across time and samples
feature_importance = np.abs(attributions).mean(axis=(0, 1))
feature_importance_normalised = feature_importance / feature_importance.sum()

# Plot top 20 features
importance_df = pd.DataFrame({
    'feature': final_features,
    'importance': feature_importance_normalised
}).sort_values('importance', ascending=True).tail(20)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(importance_df['feature'], importance_df['importance'],
        color='steelblue')
ax.set_xlabel('Mean |Attribution| (Normalised)')
ax.set_title('Top 20 Features by Integrated Gradients Attribution\n(Sepsis Cases)',
             fontweight='bold')
plt.tight_layout()
plt.savefig('images/feature_attribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Set model back to eval mode
wrapped.eval()

print("\nTop 10 most important features for sepsis prediction:")
for _, row in importance_df.tail(10).iloc[::-1].iterrows():
    print(f"  {row['feature']}: {row['importance']:.4f}")

In [ ]:
# Early warning lead time analysis
print("Analysing early warning lead time...")

# Build a DataFrame of test predictions with patient IDs
test_df_predictions = pd.DataFrame({
    'patient_id': pids_test,
    'time': times_test,
    'true_label': y_test,
    'pred_prob': lstm_prob
})

# Find first alarm time vs first sepsis time per patient
threshold = 0.5
test_df_predictions['alarm'] = test_df_predictions['pred_prob'] >= threshold

# Sample sepsis patients from test set
sepsis_test_patients = test_df_predictions[
    test_df_predictions['true_label'] == 1]['patient_id'].unique()

lead_times = []
for pid in sepsis_test_patients:
    p_data = test_df_predictions[test_df_predictions['patient_id'] == pid].sort_values('time')
    sepsis_times = p_data[p_data['true_label'] == 1]['time']
    alarm_times = p_data[p_data['alarm']]['time']

    if len(sepsis_times) > 0 and len(alarm_times) > 0:
        first_sepsis = sepsis_times.iloc[0]
        first_alarm = alarm_times.iloc[0]
        lead_time = first_sepsis - first_alarm
        if -24 < lead_time < 24:
            lead_times.append(lead_time)

fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(lead_times, bins=30, color='steelblue', edgecolor='white')
ax.axvline(x=0, color='red', linestyle='--', linewidth=2,
           label='Sepsis onset')
ax.set_xlabel('Hours Between First Alarm and Sepsis Onset\n(Positive = early warning)')
ax.set_ylabel('Number of Patients')
ax.set_title('Early Warning Lead Time Distribution', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('images/early_warning_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📊 Early Warning Statistics:")
print(f"  Patients analysed: {len(lead_times)}")
print(f"  Median lead time: {np.median(lead_times):.1f} hours")
print(f"  Mean lead time: {np.mean(lead_times):.1f} hours")
print(f"  % of alarms BEFORE sepsis onset: {(np.array(lead_times) > 0).mean():.1%}")